# 06 · Resolve customer, product and account mappings

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [ ]:
from IPython.display import display
import pandas as pd

from silver_notebook import build, frame, input_frame, settings_from_file

model = build("mdm", settings_from_file())
display(input_frame(model))

The fixture supplies seeded decisions for Customer and Product and Finance-owned classifications for local accounts. A fixture alias must resolve to one local record before a crosswalk can be published. Approved Customer fixture pairs must also agree on the recorded registration ID. Product fixture approvals use the seeded decision file. Pending pairs retain their local record and do not receive a shared ID.

Identity approval and unit conversion are separate. Review the documented opt-in approval shape in the runbook if you are exercising the dealer seal pack case. Its original quantity and amount remain on the local order line.

In [ ]:
display(frame(model, "match_candidate"))
display(frame(model, "customer_crosswalk"), frame(model, "product_crosswalk"))
display(frame(model, "account_crosswalk"), frame(model, "uom_conversion"))

In [ ]:
conformed = frame(model, "order_line_conformed")
display(conformed)
if not conformed.empty:
    coverage = conformed.assign(
        customer_resolved=conformed["group_customer_id"].notna(),
        product_resolved=conformed["group_product_id"].notna(),
    )
    display(coverage.groupby(["legal_entity_id", "customer_resolved", "product_resolved"])
            .size().rename("lines").reset_index())
display(pd.DataFrame(model.exceptions))